In [ ]:
%pip install geoai-py leafmap --q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.8/601.8 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 667.5/667.5 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1

In [ ]:
import geopandas as gpd
import leafmap
import geoai # tool to load satellite imagery data (data source: MS Planetary Computer)

import rasterio
from rasterio.merge import merge
from rasterio.windows import from_bounds # clipping tool from raster image based on coords
from rasterio.features import rasterize # This converts/transforms polygons into masks for AI models

from shapely.geometry import box # tool to create building box polygons
import glob # to handle multiple file paths (rater files)

In [ ]:
# Load vector (building polygon)
# Load OSM Building Data
gdf = gpd.read_file("/content/filename.geojson")

In [ ]:
# array([-73.7405384,  41.0362055, -73.7358419,  41.0390173])

# [min x(west), min y(south), max x(east), max y(north)]
bbox = gdf.total_bounds
bbox

array([-73.7405384,  41.0362055, -73.7358419,  41.0390173])

In [ ]:
# Load raster (satellite imagery)
collections = geoai.pc_collection_list()
# collections

items = geoai.pc_stac_search(
    collection = "naip",
    bbox = bbox,
    time_range = "2021-01-01/2023-12-31" # mostly recent two years
)
items # 2 tiles

[<Item id=ct_m_4107359_sw_18_030_20230705_20231113>,
 <Item id=ny_m_4107359_sw_18_060_20220915>,
 <Item id=ct_m_4107359_sw_18_060_20211105>]

In [ ]:
geoai.pc_item_asset_list(items[0])
# geoai.view_pc_item(item=items[0])

['image', 'thumbnail', 'tilejson', 'rendered_preview']

In [ ]:
# 2 tiles
download = geoai.pc_stac_download(
    items,
    output_dir = "data",
    assets=["image"]
)

In [ ]:
# combine/merge multiple tiles (mosaic)

files = glob.glob("data/*.tif") # merge two tiles
srcs = [rasterio.open(f) for f in files]
mosaic, out_trans = merge(srcs) # This process is known as 'mosaic'
                                # var mosaic: merged image (single image)
                                # out_trans: transform info

In [ ]:
# Adjust CRS between vector and raster
# OSM (vector) lat,lon / PC (raster) meter

# Convert bbox to raster's CRS
bbox_geom = box(bbox[0], bbox[1], bbox[2], bbox[3]) # bbox -> polygon
gdf_bbox = gpd.GeoDataFrame(geometry=[bbox_geom], crs="EPSG:4326") # create GeoDataFrame
gdf_bbox = gdf_bbox.to_crs(srcs[0].crs) # CRS adjustment
minx, miny, maxx, maxy = gdf_bbox.total_bounds
print(minx, miny, maxx, maxy)

In [ ]:
# Clip Raster
window = from_bounds(minx, miny, maxx, maxy, transform=out_trans)
clipped = mosaic[
    :,
    int(window.row_off):int(window.row_off + window.height),
    int(window.col_off):int(window.col_off + window.width),
]

In [ ]:
out_meta = srcs[0].meta.copy()

out_meta.update(
    {
        "height": clipped.shape[1],
        "width": clipped.shape[2],
        "transform": rasterio.windows.transform(window, out_trans)
    }
)

with rasterio.open("merged_clipped.tif", "w", **out_meta) as dest:
  dest.write(clipped)

In [ ]:
print(srcs[0].bounds)
print(minx, miny, maxx, maxy)

In [ ]:
# Visualize the clipped raster
m = leafmap.Map()
m.add_raster("merged_clipped.tif", layer_name="NAIP")
m

In [ ]:
# Convert entire vector data CRS to raster CRS

gdf = gpd.read_file("/content/filename.geojson")
gdf = gdf.to_crs(srcs[0].crs) # CRS of entire vector (gdf) converted to raster's CRS
gdf = gdf[gdf.geometry.type.isin(["Polygon", "Multipolygon"])]

In [ ]:
# Checking if two data are on the same CRS

# raster
with rasterio.open("/content/merged_clipped.tif") as src:
  print("Raster CRS: ", src.crs)
  print("Raster bounds: ", src.bounds)
  print("Resolution: ", src.res)

# vector
print("Vector CRS: ", gdf.crs)
print("Vector bounds: ", gdf.total_bounds)

In [ ]:
# 1. raster (NAIP)
m.add_raster("merged_clipped.tif", layer_name="NAIP")

# 2. vector (OSM Building)
m.add_gdf(
    gdf,
    layer_name="Buildings",
    style={"color": "red", "weight": 1}
)

m